# Summarize Calibration Targets

We use this scirpt to summarize mode choice calibration targets.  Underlying our analysis is a simple mode choice model where the user chooses between ride-hailing, transit and walk.  Our model applies specifically within the City of Chicago, excluding airports.  Therefore, we start by summarizing the number of trips on each mode within our target area.  Note that for ride-hailing, we have two possible data sources: the household travel survey and the Chicago TNP data.  The TNP data are an enumeration, so we prefer those over the HH survey, which is a smaller sample.  


In [14]:
import pandas as pd
pd.set_option('display.max_columns', None)
pd.reset_option('display.float_format')

from codebook import Codebook
cb = Codebook()

In [29]:
## start with just the survey data
trip_linked = pd.read_csv('data/trip_linked.csv',
    dtype={'o_tract_2020': 'Int64', 'd_tract_2020': 'Int64'})
trip_linked

,hh_id,person_id,person_num,day_id,day_num,joint_trip_id,joint_trip_num,depart_date,depart_hour,depart_minute,depart_seconds,arrive_date,arrive_hour,arrive_minute,arrive_second,distance_meters,distance_miles,duration_minutes,dwell_mins,flag_speed,flag_distance,flag_duration,o_tract_2020,d_tract_2020,hh_member_1,hh_member_2,hh_member_3,hh_member_4,hh_member_5,hh_member_6,hh_member_7,hh_member_8,hh_member_9,hh_member_10,o_purpose,o_purpose_category,d_purpose,d_purpose_category,n_legs,leg_num,first_leg,last_leg,linked_trip_id,linked_trip_num,linked_trip_mode,outbound,joint_status,linked_trip_weight,tour_id,tour_num
0,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,20,0.0,2024-05-21,11,40,0.0,1304,0.810270,20,5.0,0,0,0,17031320101,17031081500,1,0,0,0,0,0,0,0,0,0,1,1,33,10,4,1,1,0,2400012401010101,1,15,1,1,1853.792592,24000124010101,1
1,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,45,0.0,2024-05-21,12,13,0.0,544,0.338027,28,82.0,0,0,0,17031081500,17031081403,1,0,0,0,0,0,0,0,0,0,33,10,150,12,4,2,0,0,2400012401010102,2,15,0,1,1853.792592,24000124010101,1
2,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,35,0.0,2024-05-21,13,50,0.0,884,0.549293,15,9.0,0,0,0,17031081403,17031320101,1,0,0,0,0,0,0,0,0,0,150,12,33,10,4,3,0,0,2400012401010103,3,15,0,1,1853.792592,24000124010101,1
3,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,59,0.0,2024-05-21,14,15,0.0,514,0.319386,16,NaN,0,0,0,17031320101,17031320101,1,0,0,0,0,0,0,0,0,0,33,10,1,1,4,4,0,1,2400012401010104,4,15,0,1,1853.792592,24000124010101,1
4,24000124,2400012402,2,240001240201,1,-1,NaN,2024-05-21,9,15,0.0,2024-05-21,9,31,0.0,1210,0.751861,16,9.0,0,0,0,17031320101,17031320102,0,1,0,0,0,0,0,0,0,0,1,1,33,10,2,1,1,0,2400012402010101,1,15,1,1,1853.792592,24000124020101,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60057,24601170,2460117004,4,246011700401,1,2460117000102,29,2025-06-04,18,20,0.0,2025-06-04,18,30,0.0,2230,1.385661,10,10.0,0,0,0,17197880318,17197880317,1,0,1,1,0,0,0,0,0,0,51,9,102,8,4,2,0,0,2460117004010202,2,8,1,3,308.896900,24601170040102,2
60058,24601170,2460117004,4,246011700401,1,2460117000103,30,2025-06-04,18,40,0.0,2025-06-04,18,45,0.0,381,0.236743,5,30.0,0,0,0,17197880317,17197880317,1,0,1,1,0,0,0,0,0,0,102,8,99,10,4,3,0,0,2460117004010203,3,8,1,3,308.896900,24601170040102,2
60059,24601170,2460117004,4,246011700401,1,2460117000104,31,2025-06-04,19,15,0.0,2025-06-04,19,40,0.0,23964,14.890576,25,NaN,0,0,0,17197880317,17093890703,1,0,1,1,0,0,0,0,0,0,99,10,1,1,4,4,0,1,2460117004010204,4,8,0,3,308.896900,24601170040102,2
60060,24601172,2460117201,1,246011720101,1,-1,NaN,2025-06-04,8,0,0.0,2025-06-04,8,1,0.0,18884,11.734003,1,39.0,1,0,0,17097864511,17031803300,1,0,0,0,0,0,0,0,0,0,1,1,10,2,2,1,1,0,2460117201010101,1,10,1,1,1494.058248,24601172010101,1


First we want to understand what the data represent and which weights to use.  Start by looking through cmap_hts_dataset_guide.html, particularly the section on weights.  

After reviewing the documenation, it -appears- that the trips are expanded to represent typical weekday (Tue-Thu) conditions.  We want to use the linked trips, and will use the linked trip weights.  


To help make these more readable, in a way analogous to what is shown for R in the documentation, I created a phython helper class.  This usage is as follows: 

``` python 
from codebook import Codebook
cb = Codebook()

# R: factor(transit_freq, levels=..., labels=...)
person["transit_freq_f"] = cb.as_categorical(person["transit_freq"], "person", "transit_freq")

# Lighter: just swap codes for labels
person["transit_freq_label"] = cb.decode(person["transit_freq"], "person", "transit_freq")

# Lookup a variable's description / data type / skip logic
cb.describe("person", "transit_freq")

# Browse everything documented for a table
cb.variables_in("person")
```

In [32]:
# Let's look at an example summarizing the frequency of trip modes

trip_linked['linked_trip_mode_labeled'] = cb.decode(trip_linked['linked_trip_mode'], 'trip_linked', 'linked_trip_mode')

pd.DataFrame({
    'count':    trip_linked['linked_trip_mode_labeled'].value_counts(),
    'weighted': trip_linked.groupby('linked_trip_mode_labeled')['linked_trip_weight'].sum(),
}).sort_index().round()


,count,weighted
linked_trip_mode_labeled,,
Bike,1235,359491.0
Drive to bus,51,55563.0
Drive to rail,267,321613.0
HOV (2 people),11973,6838662.0
HOV (3+ people),9841,5282716.0
Long distance mode,43,22065.0
Micromobility,51,13375.0
Not imputable,1113,111951.0
Other,311,201671.0


In [33]:
# now let's recode these into a shorter set of modes:

mode_recode = {
	-1 : 'not_imputable',  # Not imputable
	1  : 'school_bus',     # School bus
	2  : 'transit',        # Drive to ferry
	3  : 'transit',        # Drive to rail
	4  : 'transit',        # Drive to bus
	5  : 'transit',        # Walk to ferry
	6  : 'transit',        # Walk to rail
	7  : 'transit',        # Walk to bus
	8  : 'car',            # HOV (3+ people)
	9  : 'car',            # HOV (2 people)
	10 : 'car',            # SOV
	11 : 'bike',           # Bike
	12 : 'bike',           # Micromobility
	13 : 'tnc',            # Taxi
	14 : 'tnc',            # TNC
	15 : 'walk',           # Walk
	16 : 'other',          # Long distance mode
	17 : 'other',          # Other
	18 : 'transit'         # Transit
}

trip_linked['mode'] = trip_linked['linked_trip_mode'].map(mode_recode)

In [34]:
# now check that it makes sense
pd.DataFrame({
    'count':    trip_linked['mode'].value_counts(),
    'weighted': trip_linked.groupby('mode')['linked_trip_weight'].sum(),
}).sort_index().round()

,count,weighted
mode,,
bike,1286,372867.0
car,42958,24928666.0
not_imputable,1113,111951.0
other,354,223736.0
school_bus,881,671795.0
tnc,853,393457.0
transit,2836,2123921.0
walk,9781,3898687.0


In [35]:
# Now let's join in the place names so we can identify which trips start and end in Chicago

# load the lookup
df_tract_place = pd.read_csv('data/tract_2020_place.csv')   
tract_to_city = dict(zip(df_tract_place['GEOID'], df_tract_place['Place']))

trip_linked['o_city'] = trip_linked['o_tract_2020'].map(tract_to_city)
trip_linked['d_city'] = trip_linked['d_tract_2020'].map(tract_to_city)

trip_linked

,hh_id,person_id,person_num,day_id,day_num,joint_trip_id,joint_trip_num,depart_date,depart_hour,depart_minute,depart_seconds,arrive_date,arrive_hour,arrive_minute,arrive_second,distance_meters,distance_miles,duration_minutes,dwell_mins,flag_speed,flag_distance,flag_duration,o_tract_2020,d_tract_2020,hh_member_1,hh_member_2,hh_member_3,hh_member_4,hh_member_5,hh_member_6,hh_member_7,hh_member_8,hh_member_9,hh_member_10,o_purpose,o_purpose_category,d_purpose,d_purpose_category,n_legs,leg_num,first_leg,last_leg,linked_trip_id,linked_trip_num,linked_trip_mode,outbound,joint_status,linked_trip_weight,tour_id,tour_num,linked_trip_mode_labeled,mode,o_city,d_city
0,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,20,0.0,2024-05-21,11,40,0.0,1304,0.810270,20,5.0,0,0,0,17031320101,17031081500,1,0,0,0,0,0,0,0,0,0,1,1,33,10,4,1,1,0,2400012401010101,1,15,1,1,1853.792592,24000124010101,1,Walk,walk,Chicago,Chicago
1,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,11,45,0.0,2024-05-21,12,13,0.0,544,0.338027,28,82.0,0,0,0,17031081500,17031081403,1,0,0,0,0,0,0,0,0,0,33,10,150,12,4,2,0,0,2400012401010102,2,15,0,1,1853.792592,24000124010101,1,Walk,walk,Chicago,Chicago
2,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,35,0.0,2024-05-21,13,50,0.0,884,0.549293,15,9.0,0,0,0,17031081403,17031320101,1,0,0,0,0,0,0,0,0,0,150,12,33,10,4,3,0,0,2400012401010103,3,15,0,1,1853.792592,24000124010101,1,Walk,walk,Chicago,Chicago
3,24000124,2400012401,1,240001240101,1,-1,NaN,2024-05-21,13,59,0.0,2024-05-21,14,15,0.0,514,0.319386,16,NaN,0,0,0,17031320101,17031320101,1,0,0,0,0,0,0,0,0,0,33,10,1,1,4,4,0,1,2400012401010104,4,15,0,1,1853.792592,24000124010101,1,Walk,walk,Chicago,Chicago
4,24000124,2400012402,2,240001240201,1,-1,NaN,2024-05-21,9,15,0.0,2024-05-21,9,31,0.0,1210,0.751861,16,9.0,0,0,0,17031320101,17031320102,0,1,0,0,0,0,0,0,0,0,1,1,33,10,2,1,1,0,2400012402010101,1,15,1,1,1853.792592,24000124020101,1,Walk,walk,Chicago,Chicago
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
60057,24601170,2460117004,4,246011700401,1,2460117000102,29,2025-06-04,18,20,0.0,2025-06-04,18,30,0.0,2230,1.385661,10,10.0,0,0,0,17197880318,17197880317,1,0,1,1,0,0,0,0,0,0,51,9,102,8,4,2,0,0,2460117004010202,2,8,1,3,308.896900,24601170040102,2,HOV (3+ people),car,NaN,NaN
60058,24601170,2460117004,4,246011700401,1,2460117000103,30,2025-06-04,18,40,0.0,2025-06-04,18,45,0.0,381,0.236743,5,30.0,0,0,0,17197880317,17197880317,1,0,1,1,0,0,0,0,0,0,102,8,99,10,4,3,0,0,2460117004010203,3,8,1,3,308.896900,24601170040102,2,HOV (3+ people),car,NaN,NaN
60059,24601170,2460117004,4,246011700401,1,2460117000104,31,2025-06-04,19,15,0.0,2025-06-04,19,40,0.0,23964,14.890576,25,NaN,0,0,0,17197880317,17093890703,1,0,1,1,0,0,0,0,0,0,99,10,1,1,4,4,0,1,2460117004010204,4,8,0,3,308.896900,24601170040102,2,HOV (3+ people),car,NaN,NaN
60060,24601172,2460117201,1,246011720101,1,-1,NaN,2025-06-04,8,0,0.0,2025-06-04,8,1,0.0,18884,11.734003,1,39.0,1,0,0,17097864511,17031803300,1,0,0,0,0,0,0,0,0,0,1,1,10,2,2,1,1,0,2460117201010101,1,10,1,1,1494.058248,24601172010101,1,SOV,car,NaN,Arlington Heights


In [36]:
# next we filter to exclude trips we don't have in our TNC data.

print("Keep only trips with origin and destination in Chicago")
trips_before = len(trip_linked)
trip_linked = trip_linked[(trip_linked['o_city'] == 'Chicago') & (trip_linked['d_city'] == 'Chicago') ]
print("Before: " + str(trips_before) + " After: " + str(len(trip_linked)))

Keep only trips with origin and destination in Chicago
Before: 60062 After: 18394


In [37]:
# summarize the modes again
pd.DataFrame({
    'count':    trip_linked['mode'].value_counts(),
    'weighted': trip_linked.groupby('mode')['linked_trip_weight'].sum(),
}).sort_index().round()

,count,weighted
mode,,
bike,1013,255309.0
car,7750,4104229.0
not_imputable,492,37549.0
other,139,52011.0
school_bus,38,15139.0
tnc,500,195265.0
transit,2042,1160505.0
walk,6420,2081707.0


Now, let's check against the Chicago TNC data.  Theoretically, it should give us the same number of TNC trips, but there is sampling error, etc.  

In [38]:
# read the initial data
tnc_trips = pd.read_csv('data/june06vot_v2.csv')
tnc_trips

,Unnamed: 0.3,Unnamed: 0.2,Unnamed: 0.1,Unnamed: 0,id,startts,endts,second,mile,pertime,perdis,p_cen,d_cen,fare_f,fare_t,fare_a,fare,share_m,share_p,sdate,edate,stime,etime,tnc_min,o,d,od,trantt,shr,odt,speed,o_zone,d_zone,od_zone,tripcount,walktt,basefare,vot
0,0,0,6951034,11831060,fab8b497c8f77737bab93181279a1b34a867b74e,06/06/2023 01:45:00 AM,06/06/2023 01:45:00 AM,281.0,1.1,1.0,1.0,1.703106e+10,1.703103e+10,5.0,0.0,2.38,7.38,False,1,2023-06-06,06/06/2023,01:45:00 AM,01:45:00 AM,4.683333,17031060900,17031031400,1703106090017031031400,1968.592402,1,17031060900170310314001,14.092527,0,0,0,4,0.366667,5.0,30.0
1,1,1,6951035,11831065,f83c9983d1806a548afd1872cd0d111ca21d167d,06/06/2023 01:45:00 AM,06/06/2023 02:00:00 AM,316.0,1.2,1.0,1.0,1.703108e+10,1.703108e+10,10.0,0.0,2.42,12.42,False,1,2023-06-06,06/06/2023,01:45:00 AM,02:00:00 AM,5.266667,17031081600,17031081300,1703108160017031081300,1419.294253,1,17031081600170310813001,13.670886,1,1,5,23,0.400000,10.0,30.0
2,2,2,6951036,11831078,f46bb54f878184af7c4bc17b06b5f6a58364d350,06/06/2023 01:45:00 AM,06/06/2023 01:45:00 AM,351.0,1.2,1.0,1.0,1.703108e+10,1.703108e+10,7.5,2.0,2.43,11.93,False,1,2023-06-06,06/06/2023,01:45:00 AM,01:45:00 AM,5.850000,17031081403,17031081700,1703108140317031081700,1847.794663,1,17031081403170310817001,12.307692,1,1,5,109,0.400000,9.5,30.0
3,3,3,6951037,11831083,f068d0c971da879afe079e7f6cd45adee0346a7b,06/06/2023 01:45:00 AM,06/06/2023 01:45:00 AM,256.0,1.0,1.0,1.0,1.703108e+10,1.703108e+10,10.0,0.0,2.42,12.42,False,1,2023-06-06,06/06/2023,01:45:00 AM,01:45:00 AM,4.266667,17031081700,17031081100,1703108170017031081100,1064.336415,1,17031081700170310811001,14.062500,1,1,5,29,0.333333,10.0,30.0
4,5,5,6951039,11831089,ecec46a0d6f073ad0636f004ed9a8819573c79c2,06/06/2023 01:45:00 AM,06/06/2023 01:45:00 AM,159.0,0.7,1.0,1.0,1.703122e+10,1.703122e+10,5.0,1.0,2.38,8.38,False,1,2023-06-06,06/06/2023,01:45:00 AM,01:45:00 AM,2.650000,17031221100,17031222700,1703122110017031222700,593.184179,1,17031221100170312227001,15.849057,0,0,0,1,0.233333,6.0,30.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
89223,111853,111853,6952607,11837151,331c9c8e91ee64c3edb117f4f6c2d7e0434c99d9,06/06/2023 12:00:00 AM,06/06/2023 12:15:00 AM,421.0,1.6,1.0,1.0,1.703108e+10,1.703108e+10,5.0,3.0,1.23,9.23,False,1,2023-06-06,06/06/2023,12:00:00 AM,12:15:00 AM,7.016667,17031081500,17031081000,1703108150017031081000,1299.136024,0,17031081500170310810000,13.681710,1,1,5,13,0.533333,8.0,10.0
89224,111854,111854,6952608,11837156,343d30494c58bde6bf82c4faec955995533d20ff,06/06/2023 12:00:00 AM,06/06/2023 12:00:00 AM,635.0,4.2,1.0,1.0,1.703184e+10,1.703132e+10,10.0,0.0,2.84,12.84,False,1,2023-06-06,06/06/2023,12:00:00 AM,12:00:00 AM,10.583333,17031841100,17031320100,1703184110017031320100,4516.695465,0,17031841100170313201000,23.811024,0,1,1,28,1.400000,10.0,30.0
89225,111855,111855,6952609,11837160,3557ee2f0fa51e07948160d51f96c279f8438a37,06/06/2023 12:00:00 AM,06/06/2023 12:00:00 AM,196.0,0.7,1.0,1.0,1.703108e+10,1.703108e+10,5.0,0.0,1.23,6.23,False,1,2023-06-06,06/06/2023,12:00:00 AM,12:00:00 AM,3.266667,17031081401,17031081201,1703108140117031081201,972.629402,0,17031081401170310812010,12.857143,1,1,5,22,0.233333,5.0,30.0
89226,111856,111856,6952610,11837162,3579b613fe5dde1325534c7143548c401434a41e,06/06/2023 12:00:00 AM,06/06/2023 12:15:00 AM,962.0,4.6,1.0,1.0,1.703183e+10,1.703108e+10,10.0,0.0,3.31,13.31,False,1,2023-06-06,06/06/2023,12:00:00 AM,12:15:00 AM,16.033333,17031832000,17031081700,1703183200017031081700,6438.160053,0,17031832000170310817000,17.214137,0,1,1,3,1.533333,10.0,30.0


Normally, I would defer to the numbers in the TNC data as an enumeration instead of a sample.  Here it seems like too few.  I want to do a bit more cross-checking before I am convinced.  